<a href="https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)



In [ ]:
#setup

import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("flyrank-ai"):
    subprocess.run(["git", "clone", "https://github.com/Farrukh776/flyrank-ai.git"], check=True)
if os.path.basename(os.getcwd()) != "flyrank-ai":
    os.chdir("flyrank-ai")

%pip -q install duckdb
import duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

features = con.sql(f"""
SELECT
  f.content_hash_id, f.client_hash_id,
  SUM(f.gsc_impressions) AS impressions_month,
  SUM(f.gsc_clicks) AS clicks_month,
  SUM(f.gsc_sum_position) / NULLIF(SUM(f.gsc_impressions), 0) AS avg_position,
  DATE '2026-03-31' - ANY_VALUE(d.content_created_date) AS content_age_days,
  ANY_VALUE(d.word_count) AS word_count,
  ANY_VALUE(d.search_volume) AS search_volume,
  ANY_VALUE(d.competition) AS competition
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet') f
LEFT JOIN read_parquet('{REL}/dim_content.parquet') d
  ON f.content_hash_id = d.content_hash_id
GROUP BY f.content_hash_id, f.client_hash_id
""").df()
features["ctr"] = features["clicks_month"] / features["impressions_month"].replace(0, np.nan)

halves = con.sql(f"""
SELECT content_hash_id,
  SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions END) AS impr_h1,
  SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions END) AS impr_h2
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
GROUP BY content_hash_id
""").df()
halves["pct_change"] = (halves["impr_h2"] - halves["impr_h1"]) / halves["impr_h1"].replace(0, pd.NA)
halves["declined"] = (halves["pct_change"] < 0).astype(int)

df = features.merge(halves[["content_hash_id", "declined"]], on="content_hash_id")

df["stale_flag"] = (df["content_age_days"] >= 365).fillna(False).astype(int)
df["visible_flag"] = (df["impressions_month"] >= 500).fillna(False).astype(int)
df["baseline_score"] = df["stale_flag"] * df["visible_flag"] * df["impressions_month"]

df = df.dropna(subset=["avg_position", "ctr"])
print(df.shape, "| decline base rate:", df["declined"].mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 14) | decline base rate: 0.377


## 1. Method choice and why

Per the training-honest-models menu, this is a "which ones first?" ranking problem with an observed (proxy) yes/no label — so the recommended path is Logistic Regression first, then Random Forest. I'll train both: Logistic Regression as the readable reference point, Random Forest as the stronger candidate, and use permutation importance on the winner to interpret what it actually leans on. I'm deliberately not reaching for Gradient Boosting — the baseline_score comparison in Week 4 showed the age signal is weak/contradictory (OPPOSITE/MIXED), so the honest question is "can any reasonable model beat a dumb rule," not "how much can I squeeze out with a heavier model." Complexity should earn its place, not be assumed.

In [ ]:
print("Features used:", ["impressions_month", "avg_position", "ctr", "content_age_days",
                          "word_count", "search_volume", "competition"])
print("Label:", "declined (within-month proxy, defined in Week 3/4)")

Features used: ['impressions_month', 'avg_position', 'ctr', 'content_age_days', 'word_count', 'search_volume', 'competition']
Label: declined (within-month proxy, defined in Week 3/4)


## 2. Split design

Grouped by client_hash_id, not a random row split. Pages from the same client share site-wide characteristics (design, niche, backlink profile) — a random split would let the model see some of a client's pages in training and others in test, silently leaking client identity as signal. GroupShuffleSplit keeps every client entirely in either train or test, which is the same standard the starter pipeline itself used (client_holdout).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["impressions_month", "avg_position", "ctr", "content_age_days",
                 "word_count", "search_volume", "competition"]
X = df[feature_cols].fillna(0)
y = df["declined"]
groups = df["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = df.iloc[test_idx]["baseline_score"].values

print(f"Train: {len(X_train)} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test)} rows, {groups.iloc[test_idx].nunique()} clients")
print("Overlap check (should be 0):", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

Train: 138310 rows, 37 clients
Test:  38428 rows, 10 clients
Overlap check (should be 0): 0


## 3. Train + compare vs my baseline



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Dummy floor
dummy = DummyClassifier(strategy="most_frequent", random_state=42).fit(X_train, y_train)
dummy_score = dummy.predict_proba(X_test)[:, 1]

# Logistic Regression
logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]

# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "method": ["base rate (random)", "dummy (majority class)", "Week-4 baseline rule",
               "Logistic Regression", "Random Forest"],
    "precision_at_20": [
        y_test.mean(),
        precision_at_k(dummy_score, y_test.values, 20),
        precision_at_k(baseline_test, y_test.values, 20),
        precision_at_k(logreg_score, y_test.values, 20),
        precision_at_k(rf_score, y_test.values, 20),
    ],
    "precision_at_50": [
        y_test.mean(),
        precision_at_k(dummy_score, y_test.values, 50),
        precision_at_k(baseline_test, y_test.values, 50),
        precision_at_k(logreg_score, y_test.values, 50),
        precision_at_k(rf_score, y_test.values, 50),
    ],
})
results

,method,precision_at_20,precision_at_50
0,base rate (random),0.402571,0.402571
1,dummy (majority class),0.100000,0.200000
2,Week-4 baseline rule,0.050000,0.100000
3,Logistic Regression,0.450000,0.440000
4,Random Forest,0.550000,0.680000


## 4. Errors and interpretation


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print(importance_df)

test_df = df.iloc[test_idx].copy()
test_df["rf_score"] = rf_score
false_positives = test_df[(test_df["rf_score"] > 0.7) & (test_df["declined"] == 0)]
print("\nFalse positives (model confident it declined, but it didn't):")
print(false_positives[["content_hash_id", "rf_score", "impressions_month", "avg_position", "content_age_days"]].head(3))

             feature  importance
3   content_age_days    0.091332
1       avg_position    0.037829
0  impressions_month    0.025265
2                ctr    0.018569
4         word_count    0.009070
5      search_volume    0.007770
6        competition    0.000673

False positives (model confident it declined, but it didn't):
Empty DataFrame
Columns: [content_hash_id, rf_score, impressions_month, avg_position, content_age_days]
Index: []


Top feature: content_age_days (permutation importance 0.091 — more than double the next feature, avg_position at 0.038), followed by impressions_month (0.025), ctr (0.019), word_count (0.009), search_volume (0.008), and competition (near zero).

This is a more interesting result than it first appears. Week 4's signal audit showed age's relationship to decline is non-monotonic: 26.2% decline for new content (<180d), dropping to 15.6% for mid-age content (180-365d), then rising again to 23.4% for old content (365d+) — a U-shape, not a straight line. The baseline rule assumed a single threshold ("365+ days = bad"), which can only express a monotonic assumption — and that's exactly why it scored below random chance. The Random Forest, by contrast, can split age at multiple points and capture the U-shape directly, which is why the same underlying signal that broke the hand rule became the model's strongest feature. The lesson isn't "age doesn't matter" — it's "age matters, but not in the shape a simple rule can express."

competition (0.0007) contributes almost nothing — a legitimately weak feature, unlike age.

In [ ]:
false_positives = test_df.sort_values("rf_score", ascending=False)
false_positives = false_positives[false_positives["declined"] == 0].head(3)
print(false_positives[["content_hash_id", "rf_score", "impressions_month", "avg_position", "content_age_days"]])

                 content_hash_id  rf_score  impressions_month  avg_position  \
10093   content_916c59ad0e5ee7bf  0.672913                1.0           8.0   
175864  content_5ba415c2fb9b6f3f  0.670434                1.0           2.0   
175856  content_d0f3f405ca3f78ed  0.670428                1.0           3.0   

        content_age_days  
10093                 46  
175864                46  
175856                46  


Where the model is wrong: the top 3 false positives (rf_score ≈ 0.67, but declined = 0) all share the same profile — impressions_month = 1.0 and content_age_days = 46 (very new, essentially untested content). This points to a genuine label-quality issue rather than a model mistake: with only 1 impression for the whole month, the within-month declined proxy (built from a first-half/second-half split) is deciding a page's fate based on which half of the month a single impression happened to land in — effectively noise, not a real trend. The model has learned that young, low-traffic pages are volatile and often flagged as "risky," which is reasonable behavior — it's the ground-truth label on these specific rows that's unreliable, not the model's judgment.

Honest takeaway: Random Forest meaningfully beats both the baseline rule and Logistic Regression (0.68 vs 0.10 vs 0.44 precision@50), and it does so by capturing a real non-monotonic relationship between content age and decline that a single-threshold rule couldn't express. Its main error mode is on very-low-traffic young content, where the proxy label itself is too noisy to trust — a real limitation of this month's data, not of the model.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.